# 15 - Empirical variable construction

This notebook constructs USDJPY empirical state variables without causal, forecasting, or event-effect analysis.

## Volatility, variance and the empirical state variables

The economic object is volatility, but the main states are variances. If $\sigma_t$ is volatility, variance is $\sigma_t^2$. One-hour log returns produce **realised variance** by squaring and summing return increments. Bloomberg supplies **implied volatility** quote points: decimal implied volatility is $q_t/100$ and annualised **implied variance** is $(q_t/100)^2$. Realised volatility is the square root of realised variance and is not the primary downstream state.

In [23]:
from pathlib import Path
import numpy as np
import pandas as pd
from zoneinfo import ZoneInfo

pd.set_option('display.max_columns', 50)
NY = ZoneInfo('America/New_York')
PRIMARY_MIN_VALID_HOURS = 24
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'Data' / 'interim').exists())
INTERIM, PROCESSED = PROJECT_ROOT / 'Data' / 'interim', PROJECT_ROOT / 'Data' / 'processed'
PROCESSED.mkdir(exist_ok=True)
print(f'Project root: {PROJECT_ROOT}')
print(f'Primary RV rule: {PRIMARY_MIN_VALID_HOURS}/24 valid hourly returns; RV is unscaled.')

Project root: c:\Users\Rajiv Nawal\OneDrive\Documents\GITREPOS\HSBC-AN-RN\CODEWORK\Linus' Task Re-Do
Primary RV rule: 24/24 valid hourly returns; RV is unscaled.


## 1. Audited inputs and frozen conventions

Notebook 14 remains the audit authority. This notebook consumes its CSV outputs without re-auditing raw data. The 1D ATM implied-volatility quote time, timezone, and exact overnight day-count remain unresolved.

In [24]:
hourly = pd.read_csv(INTERIM / 'usdjpy_hourly_mid_audited.csv', parse_dates=['timestamp', 'timestamp_parsed'])
iv = pd.read_csv(INTERIM / 'usdjpy_atm_1d_audited.csv', parse_dates=['date'])
events = pd.read_csv(INTERIM / 'g10_usd_jpy_events_audited.csv')
boj = pd.read_csv(INTERIM / 'boj_calendar_audited.csv')
inventory = pd.read_csv(INTERIM / 'empirical_source_inventory.csv')
coverage_audit = pd.read_csv(INTERIM / 'empirical_coverage_summary.csv')
quality_audit = pd.read_csv(INTERIM / 'empirical_data_quality_issues.csv')
hourly['timestamp'] = pd.to_datetime(hourly['timestamp'], utc=True)
hourly['timestamp_parsed'] = pd.to_datetime(hourly['timestamp_parsed'], utc=True)
events['event_datetime_utc'] = pd.to_datetime(events['event_datetime_utc'], utc=True, format='mixed')
events['event_date_utc'] = pd.to_datetime(events['event_date_utc'], utc=True, format='mixed')
print({'hourly_rows': len(hourly), 'iv_rows': len(iv), 'event_rows': len(events), 'boj_supplement_rows': len(boj)})
display(inventory[['source_name', 'intended_role', 'status']])
display(coverage_audit)
display(quality_audit[['Issue', 'Source', 'Status']])

{'hourly_rows': 149842, 'iv_rows': 6452, 'event_rows': 16880, 'boj_supplement_rows': 108}


,source_name,intended_role,status
0,usdjpy_hourly_mid.csv,Primary native Dukascopy 1-hour USDJPY midpoin...,Primary
1,usdjpy_hourly_bid.csv,Bid-side cross-check for hourly midpoint,Supplementary / audit
2,usdjpy_hourly_ask.csv,Ask-side cross-check for hourly midpoint,Supplementary / audit
3,ATM volatility workbook,Bloomberg PX_LAST for USDJPYVON BGN Curncy; pr...,Primary
4,G10 macro calendar,USD and JPY scheduled macro events,Primary
5,Dedicated BoJ calendar,BoJ meeting-date/timezone cross-check; not pri...,Supplementary / audit
6,Daily bid/ask/last workbook,Supplementary daily USDJPY cross-check,Supplementary / audit
7,Legacy hourly bid/ask/last workbook,Supplementary hourly USDJPY cross-check,Supplementary / audit
8,25R workbook,Broader options dataset; not required for ATM ...,Supplementary / inventory only
9,25B workbook,Broader options dataset; not required for ATM ...,Supplementary / inventory only


,source,start,end,observations,role,broad_feasible_start,broad_feasible_end,effective_start_driver
0,New hourly USDJPY midpoint,2003-05-04,2026-07-01,149842,Primary intraday spot,2003-05-04,2026-07-01,hourly USDJPY
1,Legacy daily USDJPY,1986-06-10,2026-06-10,10437,Supplementary spot,2003-05-04,2026-07-01,hourly USDJPY
2,USDJPY 1D ATM IV,1998-12-14,2026-07-01,6452,Primary daily IV,2003-05-04,2026-07-01,hourly USDJPY
3,USD macro events,1997-01-02,2026-07-01,12130,Primary event calendar subset,2003-05-04,2026-07-01,hourly USDJPY
4,JPY macro events,1997-01-13,2026-07-01,4750,Primary event calendar subset,2003-05-04,2026-07-01,hourly USDJPY
5,Dedicated BoJ events (UTC),2015-01-21,2026-06-15,108,Supplementary BoJ meeting-date/timezone cross-...,2003-05-04,2026-07-01,hourly USDJPY


,Issue,Source,Status
0,Hourly sampling gaps,New hourly midpoint,Classified
1,Populated weekend / stale timestamps,New hourly midpoint,Observed
2,Major internal hourly outage,New hourly midpoint,Observed
3,Supplied-return-across-gap risk,New hourly midpoint,Flagged
4,Hourly timestamp timezone,New hourly midpoint,Resolved from source strings
5,Hourly OHLC / spread integrity,New hourly midpoint and bid/ask files,Audited
6,1D ATM quote cut/timezone,Bloomberg ATM PX_LAST,Partially resolved
7,1D ATM horizon / total-variance convention,Bloomberg ATM PX_LAST,Unresolved
8,Extraction reproducibility metadata,Dukascopy / Bloomberg,Partially unresolved
9,Long 1D ATM date gaps,ATM workbook,Audited


## 2. The 17:00 New York model day

A model day is labelled by the date on which it **ends**. If $c_t$ is that date's 17:00 New York cut, define its start as $s_t=c_t-1\text{ calendar day}$ and its session as $[s_t,c_t)$; the start is included and the end excluded. For Monday, $s_{\text{Monday}}$ is Sunday 17:00 New York and $c_{\text{Monday}}$ is Monday 17:00 New York. Boundaries are constructed in `America/New_York` then converted to UTC, so daylight-saving time is handled dynamically rather than by a hard-coded UTC hour.

| Model day | Model session begins | Model session ends | First hourly candle |
| --- | --- | --- | --- |
| Monday | Sunday 17:00 New York | Monday 17:00 New York | Sunday 17:00-18:00 |
| Tuesday | Monday 17:00 New York | Tuesday 17:00 New York | Monday 17:00-18:00 |
| Wednesday | Tuesday 17:00 New York | Wednesday 17:00 New York | Tuesday 17:00-18:00 |
| Thursday | Wednesday 17:00 New York | Thursday 17:00 New York | Wednesday 17:00-18:00 |
| Friday | Thursday 17:00 New York | Friday 17:00 New York | Thursday 17:00-18:00 |

A Dukascopy row timestamped $h$ represents approximately $[h,h+1\text{ hour})$. Thus a 09:00 candle spans 09:00 up to, but not including, 10:00; `mid_open` is its beginning price and `mid_close` is approximately its ending price. The final candle of a model day begins one hour before the 17:00 New York cut.

In [25]:
def model_day_from_candle_start(ts):
    ts = pd.Timestamp(ts)
    if ts.tzinfo is None:
        ts = ts.tz_localize('UTC')
    local = ts.tz_convert(NY)
    day = local.normalize() + pd.Timedelta(days=int((local.hour, local.minute, local.second, local.microsecond) >= (17, 0, 0, 0)))
    return day.date() if day.weekday() < 5 else pd.NaT

def make_sessions(first_day, last_day):
    days = pd.bdate_range(first_day, last_day)
    end_ny = pd.DatetimeIndex([pd.Timestamp(d.date()).tz_localize(NY) + pd.Timedelta(hours=17) for d in days])
    out = pd.DataFrame({'model_day': pd.to_datetime(days.date), 'session_end_ny': end_ny})
    out['session_start_ny'] = out['session_end_ny'] - pd.DateOffset(days=1)
    out['session_start_utc'] = out['session_start_ny'].dt.tz_convert('UTC')
    out['session_end_utc'] = out['session_end_ny'].dt.tz_convert('UTC')
    out['expected_hourly_return_count'] = ((out['session_end_utc'] - out['session_start_utc']).dt.total_seconds() / 3600).astype(int)
    return out

hourly['model_day'] = hourly['timestamp'].map(model_day_from_candle_start)
first_day = pd.to_datetime(hourly['model_day'].dropna()).min().date()
last_day = pd.to_datetime(hourly['model_day'].dropna()).max().date()
sessions = make_sessions(first_day, last_day)
assert (sessions['session_start_ny'].dt.hour == 17).all() and (sessions['session_end_ny'].dt.hour == 17).all()
assert (sessions['session_start_utc'].iloc[1:].to_numpy() >= sessions['session_end_utc'].iloc[:-1].to_numpy()).all()
ordinary = sessions['model_day'].iloc[1:].dt.weekday.ne(0).to_numpy()
assert (sessions['session_start_utc'].iloc[1:].to_numpy()[ordinary] == sessions['session_end_utc'].iloc[:-1].to_numpy()[ordinary]).all()
probe = sessions.iloc[100]
assert model_day_from_candle_start(probe['session_end_utc']) == sessions.iloc[101]['model_day'].date()
print(f'{len(sessions):,} sessions: {sessions.model_day.min().date()} to {sessions.model_day.max().date()}')

6,044 sessions: 2003-05-05 to 2026-07-02


## 3. Hourly log-return construction

For later hours in a model day, the code uses consecutive candle closes:

$$r_{t,h}=\log\left(\frac{C_{t,h}}{C_{t,h-1}}\right),\qquad h=2,\ldots,24.$$

It constructs this only for genuinely adjacent candles in the same session: a missing multi-hour interval is not bridged.

### First hour of every model day

The first within-session return is open-to-close within the new candle,

$$r_{t,1}=\log\left(\frac{C_{t,1}}{O_{t,1}}\right),$$

not $\log(C_{t,1}/C_{t-1,24})$. This prevents movement between the old session endpoint and the new opening from being mixed into the first trading-hour return. Hence a complete day contains $r_{t,1},r_{t,2},\ldots,r_{t,24}$.

*Illustrative example only:* if the first candle opens at 147.20 and closes at 147.35, then $\log(147.35/147.20)\approx0.001018$, about a 0.102% log return during that first trading hour.

In [26]:
hourly = hourly.sort_values('timestamp').reset_index(drop=True)
hourly['candle_end_utc'] = hourly['timestamp'] + pd.Timedelta(hours=1)
hourly['timestamp_ny'] = hourly['timestamp'].dt.tz_convert(NY)
hourly['model_day'] = hourly['timestamp'].map(model_day_from_candle_start)
hourly['model_day'] = pd.to_datetime(hourly['model_day'])
for c in ['mid_open', 'mid_high', 'mid_low', 'mid_close']:
    hourly[c] = pd.to_numeric(hourly[c], errors='coerce')
hourly['audit_row_valid'] = hourly[['mid_open', 'mid_high', 'mid_low', 'mid_close']].gt(0).all(axis=1)
hourly['previous_timestamp'] = hourly['timestamp'].shift()
hourly['previous_mid_close'] = hourly['mid_close'].shift()
hourly['adjacent_previous_hour'] = hourly['timestamp'].diff().eq(pd.Timedelta(hours=1))
hourly['is_session_first_candle'] = hourly['timestamp'].isin(set(sessions['session_start_utc']))
hourly['hourly_log_return'] = np.nan
hourly['return_reason'] = 'outside_model_session_or_invalid_audit'
continuation = hourly['model_day'].notna() & hourly['audit_row_valid'] & hourly['audit_row_valid'].shift(fill_value=False) & hourly['adjacent_previous_hour'] & hourly['model_day'].eq(hourly['model_day'].shift())
opening = hourly['model_day'].notna() & hourly['audit_row_valid'] & hourly['is_session_first_candle']
hourly.loc[continuation, 'hourly_log_return'] = np.log(hourly.loc[continuation, 'mid_close'] / hourly.loc[continuation, 'previous_mid_close'])
hourly.loc[continuation, 'return_reason'] = 'adjacent_within_session_close_to_close'
hourly.loc[opening, 'hourly_log_return'] = np.log(hourly.loc[opening, 'mid_close'] / hourly.loc[opening, 'mid_open'])
hourly.loc[opening, 'return_reason'] = 'session_open_to_close'
hourly.loc[opening & pd.to_datetime(hourly['model_day']).dt.weekday.eq(0), 'return_reason'] = 'first_post_weekend_session_open_to_close'
missing_previous = hourly['model_day'].notna() & hourly['audit_row_valid'] & ~hourly['is_session_first_candle'] & ~continuation
hourly.loc[missing_previous, 'return_reason'] = 'invalid_missing_previous_hour'
hourly['valid_within_session_return'] = hourly['hourly_log_return'].notna()
hourly['squared_within_session_return'] = hourly['hourly_log_return'].pow(2)
assert hourly.loc[hourly['valid_within_session_return'], 'return_reason'].isin(['adjacent_within_session_close_to_close', 'session_open_to_close', 'first_post_weekend_session_open_to_close']).all()
print(hourly['return_reason'].value_counts().to_string())

return_reason
adjacent_within_session_close_to_close      133877
outside_model_session_or_invalid_audit       10076
session_open_to_close                         4669
first_post_weekend_session_open_to_close      1140
invalid_missing_previous_hour                   80


## 4. Completeness, realised variance, and weekend reopening movements

### The 24/24 completeness rule

One model session has 24 hourly intervals. Let $N_t$ be the valid within-session increments. `rv_eligible` means eligibility for the canonical realised-variance sample and requires $N_t=24$. If 23 increments are present, $\sum_{h=1}^{23}r_{t,h}^2$ omits variation during the missing hour and is not a complete-session observation.

The notebook deliberately does not use $\frac{24}{23}\sum_{h=1}^{23}r_{t,h}^2$: the missing hour need not have representative variation. It also does not interpolate prices, set missing returns to zero, or bridge gaps. Approximately 95.37% of saved model sessions satisfy 24/24; relaxing the rule adds little coverage.

### Return versus realised variance

The signed within-session log return is $r_t^{within}=\sum_{h=1}^{24}r_{t,h}$, which for a complete session is $\log(P_t^{end}/P_t^{open})$. Within-session realised variance is $\sum_{h=1}^{24}r_{t,h}^2$: it measures accumulated variation, not direction, and uses a sum rather than an average because the object is total session variation.

For illustration, $r_1=+0.01$ and $r_2=-0.01$ have net return zero but squared-return sum $0.0001+0.0001=0.0002$. An exchange rate can finish near its start after substantial intraday movement.

`rv_within_raw` and `within_session_log_return_raw` retain available unscaled diagnostic sums. Canonical `rv_within` (within-session realised variance) and `within_session_log_return` equal their raw counterparts only when `rv_eligible` is true; otherwise they are missing.

For Monday, the closure gap is $g_t=\log(P_S^{open}/P_F^{end})$, from Friday's model-session endpoint to the Sunday 17:00 New York reopening open. The first trading-hour return remains separate. For example, $146.80\rightarrow147.20\rightarrow147.35$ separates Friday endpoint to reopening from reopening to the first candle close.

In [27]:
inside = hourly.loc[hourly['model_day'].notna()].copy()
inside['model_day'] = pd.to_datetime(inside['model_day'])
aggregated = inside.groupby('model_day', as_index=False).agg(
    observed_hourly_count=('timestamp', 'size'),
    valid_hourly_return_count=('valid_within_session_return', 'sum'),
    within_session_log_return_raw=('hourly_log_return', lambda x: x.sum(min_count=1)),
    rv_within_raw=('squared_within_session_return', lambda x: x.sum(min_count=1)),
)
daily = sessions.merge(aggregated, on='model_day', how='left')
for c in ['observed_hourly_count', 'valid_hourly_return_count']:
    daily[c] = daily[c].fillna(0).astype(int)
daily['valid_hourly_count'] = daily['valid_hourly_return_count']
daily['expected_hourly_count'] = daily['expected_hourly_return_count']
daily['coverage_ratio'] = daily['valid_hourly_return_count'] / daily['expected_hourly_return_count']

# Exact start open and exact endpoint close: endpoint candle is [end-1h, end).
starts = hourly[['timestamp', 'mid_open', 'audit_row_valid']].rename(columns={'timestamp': 'session_start_utc', 'mid_open': 'session_open_price', 'audit_row_valid': 'session_open_valid'})
ends = hourly[['timestamp', 'mid_close', 'audit_row_valid']].rename(columns={'timestamp': 'endpoint_candle_start_utc', 'mid_close': 'session_end_price', 'audit_row_valid': 'session_end_valid'})
daily['endpoint_candle_start_utc'] = daily['session_end_utc'] - pd.Timedelta(hours=1)
daily = daily.merge(starts, on='session_start_utc', how='left').merge(ends, on='endpoint_candle_start_utc', how='left')
daily['session_open_valid'] = daily['session_open_valid'].astype('boolean').fillna(False).astype(bool)
daily['session_end_valid'] = daily['session_end_valid'].astype('boolean').fillna(False).astype(bool)
daily['previous_model_day'] = daily['model_day'].shift()
daily['previous_session_end_price'] = daily['session_end_price'].shift()
daily['previous_session_end_valid'] = daily['session_end_valid'].shift(fill_value=False)
daily['session_boundary_gap_log_return'] = np.where(
    daily['session_open_valid'] & daily['previous_session_end_valid'],
    np.log(daily['session_open_price'] / daily['previous_session_end_price']), np.nan
)
daily['is_monday'] = daily['model_day'].dt.weekday.eq(0)
daily['eligible_monday_session'] = daily['is_monday'] & daily['previous_model_day'].dt.weekday.eq(4)
daily['weekend_gap_missing_reason'] = pd.NA
weekend_ok = daily['eligible_monday_session'] & daily['session_open_valid'] & daily['previous_session_end_valid']
daily['closure_gap_type'] = np.where(weekend_ok, 'weekend', np.where(daily['eligible_monday_session'], 'unresolved_data_gap', 'none'))
daily.loc[daily['eligible_monday_session'] & ~weekend_ok & ~daily['previous_session_end_valid'] & ~daily['session_open_valid'], 'weekend_gap_missing_reason'] = 'Friday endpoint and Sunday reopening unavailable'
daily.loc[daily['eligible_monday_session'] & ~weekend_ok & ~daily['previous_session_end_valid'] & daily['session_open_valid'], 'weekend_gap_missing_reason'] = 'Friday endpoint unavailable'
daily.loc[daily['eligible_monday_session'] & ~weekend_ok & daily['previous_session_end_valid'] & ~daily['session_open_valid'], 'weekend_gap_missing_reason'] = 'Sunday 17:00 reopening candle unavailable'
daily['closure_gap_log_return'] = np.where(weekend_ok, daily['session_boundary_gap_log_return'], np.where(daily['eligible_monday_session'], np.nan, 0.0))
# Independent endpoint-to-endpoint construction; never assigned from component returns.
daily['total_cut_to_cut_log_return'] = np.where(
    daily['session_end_valid'] & daily['previous_session_end_valid'],
    np.log(daily['session_end_price'] / daily['previous_session_end_price']), np.nan
)
daily['rv_with_closure_gap_raw'] = np.where(
    daily['rv_within_raw'].notna() & daily['closure_gap_log_return'].notna(),
    daily['rv_within_raw'] + daily['closure_gap_log_return'].pow(2), np.nan
)
daily['rv_eligible'] = daily['valid_hourly_return_count'].ge(PRIMARY_MIN_VALID_HOURS)
# Canonical modelling quantities are masked to complete 24/24 sessions; raw values remain diagnostic only.
daily['within_session_log_return'] = daily['within_session_log_return_raw'].where(daily['rv_eligible'])
daily['rv_within'] = daily['rv_within_raw'].where(daily['rv_eligible'])
daily['rv_with_closure_gap'] = daily['rv_with_closure_gap_raw'].where(daily['rv_eligible'])

# Attach the corrected boundary classification only to exact session-opening candles in the hourly export.
hourly['closure_gap_type'] = 'none'
hourly['closure_gap_log_return'] = np.nan
opening_daily = daily[['model_day', 'closure_gap_type', 'closure_gap_log_return']].copy()
hourly = hourly.merge(opening_daily, on='model_day', how='left', suffixes=('', '_daily'))
opening_rows = hourly['is_session_first_candle']
hourly.loc[opening_rows, 'closure_gap_type'] = hourly.loc[opening_rows, 'closure_gap_type_daily'].fillna('none')
hourly.loc[opening_rows, 'closure_gap_log_return'] = hourly.loc[opening_rows, 'closure_gap_log_return_daily']
hourly = hourly.drop(columns=['closure_gap_type_daily', 'closure_gap_log_return_daily'])
assert (daily.loc[daily['rv_eligible'], 'valid_hourly_return_count'] == 24).all()
assert daily.loc[~daily['rv_eligible'], 'rv_within'].isna().all()
assert daily.loc[daily['rv_eligible'], 'rv_within'].notna().all()
assert np.allclose(daily.loc[daily['rv_eligible'], 'rv_within'], daily.loc[daily['rv_eligible'], 'rv_within_raw'])
assert daily.loc[~daily['rv_eligible'], 'within_session_log_return'].isna().all()
assert np.allclose(daily.loc[daily['rv_eligible'], 'within_session_log_return'], daily.loc[daily['rv_eligible'], 'within_session_log_return_raw'])
assert daily.loc[~daily['rv_eligible'], 'rv_with_closure_gap'].isna().all()
assert np.allclose(daily.loc[weekend_ok, 'closure_gap_log_return'], daily.loc[weekend_ok, 'session_boundary_gap_log_return'])
assert not daily.loc[daily['model_day'].between('2010-01-01', '2010-10-01'), 'closure_gap_type'].eq('weekend').any()
print(f'Complete-day RV retention: {daily.rv_eligible.sum():,}/{len(daily):,} ({daily.rv_eligible.mean():.1%})')

Complete-day RV retention: 5,764/6,044 (95.4%)


## Session-boundary gap, cut-to-cut return, and closure-gap sensitivity

The boundary movement is $b_t=\log(P_t^{open}/P_{t-1}^{end})$ and the within-session movement is $r_t^{within}=\log(P_t^{end}/P_t^{open})$. The independently constructed cut-to-cut return is $r_t^{cut-to-cut}=\log(P_t^{end}/P_{t-1}^{end})$. Logarithmic additivity gives

$$\log\left(\frac{P_t^{open}}{P_{t-1}^{end}}\right)+\log\left(\frac{P_t^{end}}{P_t^{open}}\right)=\log\left(\frac{P_t^{end}}{P_{t-1}^{end}}\right).$$

The notebook verifies this endpoint-based equality numerically rather than defining cut-to-cut return as a sum. For Monday, the boundary movement is the weekend reopening movement when valid boundary prices exist; unexplained outages are not classified as closures.

The sensitivity `rv_with_closure_gap_raw` is raw within-session realised variance plus $g_t^2$, then `rv_with_closure_gap` is masked by eligibility. This is not $(g_t+r_t^{within})^2$: realised variance squares each distinct return increment, so a discrete weekend movement contributes its own $g_t^2$. Primary realised variance remains `rv_within`.

In [28]:
weekend_year = daily.loc[daily['eligible_monday_session']].assign(year=lambda x: x['model_day'].dt.year).groupby('year', as_index=False).agg(
    eligible_monday_sessions=('eligible_monday_session', 'size'),
    weekend_gap_constructed=('closure_gap_type', lambda x: (x == 'weekend').sum()),
    weekend_gap_missing=('closure_gap_type', lambda x: (x != 'weekend').sum()),
)
print('Weekend-gap validation by year:')
display(weekend_year)
missing_reasons = daily.loc[daily['eligible_monday_session'] & daily['closure_gap_type'].ne('weekend'), 'weekend_gap_missing_reason'].fillna('other/unresolved').value_counts().rename_axis('reason').reset_index(name='count')
print('Missing eligible-Monday gap reasons:')
display(missing_reasons)
weekend_stats = daily.loc[daily['closure_gap_type'].eq('weekend'), 'closure_gap_log_return'].agg(['count', 'mean', 'median', 'std', 'min', 'max'])
print('Weekend-gap descriptive diagnostics:')
print(weekend_stats.to_string())
rv_year = daily.assign(year=daily['model_day'].dt.year).groupby('year', as_index=False).agg(model_days=('model_day', 'size'), rv_eligible_days=('rv_eligible', 'sum'), mean_valid_hourly_return_count=('valid_hourly_return_count', 'mean'))
rv_year['rv_eligible_pct'] = rv_year['rv_eligible_days'] / rv_year['model_days']
print('RV coverage by year:')
display(rv_year)

Weekend-gap validation by year:


,year,eligible_monday_sessions,weekend_gap_constructed,weekend_gap_missing
0,2003,34,32,2
1,2004,52,51,1
2,2005,52,52,0
3,2006,52,52,0
4,2007,53,53,0
5,2008,52,52,0
6,2009,52,52,0
7,2010,52,13,39
8,2011,52,52,0
9,2012,53,52,1


Missing eligible-Monday gap reasons:


,reason,count
0,Friday endpoint and Sunday reopening unavailable,40
1,Sunday 17:00 reopening candle unavailable,6
2,Friday endpoint unavailable,5


Weekend-gap descriptive diagnostics:
count     1157.000000
mean        -0.000146
median      -0.000110
std          0.001968
min         -0.012048
max          0.013744
RV coverage by year:


,year,model_days,rv_eligible_days,mean_valid_hourly_return_count,rv_eligible_pct
0,2003,173,167,23.878613,0.965318
1,2004,262,259,23.938931,0.988550
2,2005,260,258,23.957692,0.992308
3,2006,260,259,23.969231,0.996154
4,2007,261,260,23.969349,0.996169
5,2008,262,261,23.969466,0.996183
6,2009,261,260,23.969349,0.996169
7,2010,261,64,5.996169,0.245211
8,2011,260,259,23.969231,0.996154
9,2012,261,249,23.842912,0.954023


## 5. From implied volatility to implied variance

Bloomberg supplies an **implied-volatility** quote. If $q_t=10$ quote points, decimal implied volatility is $q_t/100=0.10$. `atm_vol_quote_points` stores the original Bloomberg quote $q_t$, `implied_vol_decimal` stores decimal implied volatility $q_t/100$, and `iv_ann_var` stores annualised implied variance $(q_t/100)^2$; for 10 points this is $(0.10)^2=0.01$.

This is an annualised variance rate, whereas primary realised variance is accumulated over one model session; the two are not initially in identical time units. `iv_var_252` and `iv_var_365` therefore retain annualised implied variance divided by 252 and 365 as trading-day and calendar-day scaling sensitivities. Neither is claimed to be Bloomberg's exact overnight convention. Quote time, timezone, expiry horizon, weekend/holiday treatment, and day-count convention remain unresolved.

In [29]:
iv = iv.rename(columns={'date': 'model_day', 'iv': 'atm_vol_quote_points'}).copy()
iv['model_day'] = pd.to_datetime(iv['model_day'])
iv['atm_vol_quote_points'] = pd.to_numeric(iv['atm_vol_quote_points'], errors='coerce')
iv['implied_vol_decimal'] = iv['atm_vol_quote_points'] / 100.0
iv['iv_ann_var'] = iv['implied_vol_decimal'].pow(2)
iv['iv_var_252'] = iv['iv_ann_var'] / 252.0
iv['iv_var_365'] = iv['iv_ann_var'] / 365.0
iv['iv_timestamp_status'] = 'date_only_snapshot_unresolved'
# Backward-compatible aliases retained for existing downstream inspection.
iv['iv_quote_points'] = iv['atm_vol_quote_points']
iv['iv_ann_vol_decimal'] = iv['implied_vol_decimal']
iv['iv_var_per_252_sensitivity'] = iv['iv_var_252']
iv['iv_var_per_365_sensitivity'] = iv['iv_var_365']
assert np.allclose(iv['iv_ann_var'].dropna(), iv.loc[iv['iv_ann_var'].notna(), 'implied_vol_decimal'].pow(2))

## 6. Exact event-time assignment

Events are assigned by exact timezone-aware UTC timestamps, not calendar dates alone: $\text{session start}_t\le\tau_e<\text{session end}_t$. Events outside the empirical model-sample period are classified separately and are not treated as market-closure observations. In-sample events whose exact timestamps fall outside the active model sessions are flagged as occurring during a market closure.

Future scheduled-event indicators use ex-ante calendar time and currency only. Actual values, surprises, revisions, and other post-release fields never enter them. Same-date events are not labelled before or after the Bloomberg implied-volatility quote because the exact quote time remains unresolved.

In [30]:
session_lookup = sessions[['model_day', 'session_start_utc', 'session_end_utc']].sort_values('session_start_utc').copy()
session_lookup['forecast_origin_model_day'] = session_lookup['model_day'].shift()
sample_start, sample_end = session_lookup['session_start_utc'].iloc[0], session_lookup['session_end_utc'].iloc[-1]
events_mapped = pd.merge_asof(events.sort_values('event_datetime_utc').copy(), session_lookup, left_on='event_datetime_utc', right_on='session_start_utc', direction='backward')
events_mapped['event_outside_model_sample'] = events_mapped['event_datetime_utc'].lt(sample_start) | events_mapped['event_datetime_utc'].ge(sample_end)
events_mapped['event_in_rv_session'] = ~events_mapped['event_outside_model_sample'] & events_mapped['session_end_utc'].notna() & events_mapped['event_datetime_utc'].lt(events_mapped['session_end_utc'])
events_mapped['event_in_market_closure'] = ~events_mapped['event_outside_model_sample'] & ~events_mapped['event_in_rv_session']
events_mapped.loc[~events_mapped['event_in_rv_session'], ['model_day', 'forecast_origin_model_day']] = pd.NaT
events_mapped = events_mapped.rename(columns={'model_day': 'rv_model_day'})
events_mapped['event_status'] = np.select(
    [events_mapped['event_in_rv_session'], events_mapped['event_in_market_closure'], events_mapped['event_outside_model_sample']],
    ['mapped to RV session', 'market closure within sample', 'outside model sample'], default='other/unresolved'
)
assert len(events_mapped) == len(events)
assert not events_mapped.loc[events_mapped['event_outside_model_sample'], 'event_in_market_closure'].any()
assert events_mapped.loc[events_mapped['event_in_market_closure'], 'rv_model_day'].isna().all()
assert (events_mapped.loc[events_mapped['event_in_rv_session'], 'event_datetime_utc'] >= events_mapped.loc[events_mapped['event_in_rv_session'], 'session_start_utc']).all()
assert (events_mapped.loc[events_mapped['event_in_rv_session'], 'event_datetime_utc'] < events_mapped.loc[events_mapped['event_in_rv_session'], 'session_end_utc']).all()
event_status = events_mapped['event_status'].value_counts().reindex(['mapped to RV session', 'market closure within sample', 'outside model sample', 'other/unresolved'], fill_value=0).rename_axis('event status').reset_index(name='count')
print('Event mapping status:')
display(event_status)
print({'total_event_rows': len(events_mapped), 'in_sample_event_rows': int((~events_mapped.event_outside_model_sample).sum()), 'pre_sample_event_rows': int(events_mapped.event_datetime_utc.lt(sample_start).sum()), 'post_sample_event_rows': int(events_mapped.event_datetime_utc.ge(sample_end).sum())})

Event mapping status:


,event status,count
0,mapped to RV session,14813
1,market closure within sample,2
2,outside model sample,2065
3,other/unresolved,0


{'total_event_rows': 16880, 'in_sample_event_rows': 14815, 'pre_sample_event_rows': 2065, 'post_sample_event_rows': 0}


## 7. Forward implied variance and eligibility flags

The documented primary alignment is **implied variance at model day $t$ to realised variance at the next model day**. The realised variance dated $t$ describes the session ending at the date-$t$ 17:00 New York cut; the date-$t$ implied-variance observation is treated as forward-looking. Friday maps to Monday, not Saturday. Exact Bloomberg micro-timing is unresolved.

`joint_panel_eligible` requires complete current realised variance, available implied volatility, and event-calendar availability. It does not depend on future data. `forecast_pair_eligible` additionally requires valid realised variance for the next model day. Current-state dependence analysis should not remove a valid current observation merely because the following high-frequency session is incomplete; forecast evaluation genuinely needs that next-day target.

In [31]:
panel = daily.merge(iv.drop(columns=['date_diff', 'prior_iv_gap_over_3_days'], errors='ignore'), on='model_day', how='left')
panel['iv_available'] = panel['iv_ann_var'].notna()
panel['next_model_day'] = panel['model_day'].shift(-1)
panel['rv_next_model_day'] = panel['rv_within'].shift(-1)
panel['rv_with_closure_gap_next_model_day'] = panel['rv_with_closure_gap'].shift(-1)
panel['rv_next_model_day_eligible'] = panel['rv_eligible'].shift(-1).fillna(False).astype(bool)

# Counts use scheduled timestamp/currency only; no post-release columns enter this aggregation.
target_counts = events_mapped.loc[events_mapped['event_in_rv_session']].groupby('rv_model_day', as_index=False).agg(
    any_scheduled_event_next_session=('event', lambda x: bool(len(x))),
    scheduled_event_count_next_session=('event', 'size'),
    scheduled_us_event_count_next_session=('ccy', lambda x: (x.astype(str).str.upper() == 'USD').sum()),
    scheduled_jp_event_count_next_session=('ccy', lambda x: x.astype(str).str.upper().isin(['JPY', 'JP']).sum()),
)
origins = session_lookup[['model_day', 'forecast_origin_model_day']].rename(columns={'model_day': 'rv_model_day'})
event_origins = origins.merge(target_counts, on='rv_model_day', how='left').drop(columns='rv_model_day')
panel = panel.merge(event_origins, left_on='model_day', right_on='forecast_origin_model_day', how='left').drop(columns='forecast_origin_model_day')
for c in ['scheduled_event_count_next_session', 'scheduled_us_event_count_next_session', 'scheduled_jp_event_count_next_session']:
    panel[c] = panel[c].fillna(0).astype(int)
panel['any_scheduled_event_next_session'] = panel['any_scheduled_event_next_session'].astype('boolean').fillna(False).astype(bool)
calendar_min = events_mapped['event_datetime_utc'].min().tz_localize(None).normalize()
calendar_max = events_mapped['event_datetime_utc'].max().tz_localize(None).normalize()
panel['event_data_available'] = panel['model_day'].between(calendar_min, calendar_max)
# Current-state eligibility intentionally excludes future-target availability.
panel['joint_panel_eligible'] = panel['rv_eligible'] & panel['iv_available'] & panel['event_data_available']
panel['forecast_pair_eligible'] = panel['joint_panel_eligible'] & panel['rv_next_model_day_eligible']

comparable = panel['rv_eligible'] & panel['session_boundary_gap_log_return'].notna() & panel['total_cut_to_cut_log_return'].notna()
panel['cut_to_cut_decomposition_abs_diff'] = np.where(comparable, np.abs(panel['total_cut_to_cut_log_return'] - (panel['session_boundary_gap_log_return'] + panel['within_session_log_return'])), np.nan)
cut_stats = panel.loc[comparable, 'cut_to_cut_decomposition_abs_diff'].agg(['count', 'max', 'median', lambda x: x.quantile(0.99)])
cut_stats.index = ['comparable_eligible_sessions', 'max_abs_difference', 'median_abs_difference', 'p99_abs_difference']
assert (panel.loc[comparable, 'cut_to_cut_decomposition_abs_diff'] < 1e-10).all()
assert pd.isna(panel.iloc[-1]['next_model_day']) and pd.isna(panel.iloc[-1]['rv_next_model_day'])
assert panel['joint_panel_eligible'].eq(panel['rv_eligible'] & panel['iv_available'] & panel['event_data_available']).all()
assert panel['forecast_pair_eligible'].eq(panel['joint_panel_eligible'] & panel['rv_next_model_day_eligible']).all()
assert panel.loc[~panel['rv_next_model_day_eligible'], 'rv_next_model_day'].isna().all()
assert panel.loc[~panel['rv_next_model_day_eligible'], 'rv_with_closure_gap_next_model_day'].isna().all()
print('Independent cut-to-cut decomposition check:')
print(cut_stats.to_string())
availability = pd.Series({
    'RV available only': int((panel.rv_eligible & ~panel.iv_available).sum()),
    'IV available only': int((~panel.rv_eligible & panel.iv_available).sum()),
    'both RV and IV available': int((panel.rv_eligible & panel.iv_available).sum()),
    'neither': int((~panel.rv_eligible & ~panel.iv_available).sum()),
})
print('RV/IV availability:')
print(availability.to_string())
eligibility_report = pd.Series({
    'RV eligible': int(panel['rv_eligible'].sum()),
    'IV available': int(panel['iv_available'].sum()),
    'Current joint panel eligible': int(panel['joint_panel_eligible'].sum()),
    'Forecast pair eligible': int(panel['forecast_pair_eligible'].sum()),
    'Current joint lost solely because next RV unavailable': int((panel['joint_panel_eligible'] & ~panel['forecast_pair_eligible']).sum()),
})
print('Eligibility bookkeeping:')
print(eligibility_report.to_string())

C:\Users\Rajiv Nawal\AppData\Local\Temp\ipykernel_15080\2988867451.py:6: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  panel['rv_next_model_day_eligible'] = panel['rv_eligible'].shift(-1).fillna(False).astype(bool)


Independent cut-to-cut decomposition check:
comparable_eligible_sessions    5.744000e+03
max_abs_difference              1.030426e-15
median_abs_difference           1.779718e-16
p99_abs_difference              6.481908e-16
RV/IV availability:
RV available only            319
IV available only            274
both RV and IV available    5445
neither                        6
Eligibility bookkeeping:
RV eligible                                              5764
IV available                                             5719
Current joint panel eligible                             5445
Forecast pair eligible                                   5372
Current joint lost solely because next RV unavailable      73


## 8. Export and construction conclusion

Raw realised quantities are retained for auditability, while canonical within-session returns and realised-variance variables are masked to strict 24/24 eligibility. `joint_panel_eligible` is current information; `forecast_pair_eligible` additionally requires the subsequent realised-variance target.

The notebook establishes 17:00 New York sessions, separately measured weekend reopening movement, realised-variance sensitivity diagnostics, annualised implied variance with 252/365 sensitivities, exact event-time assignment, and ex-ante scheduled-event indicators. Bloomberg implied-volatility quote time and true overnight day-count remain unresolved.

In [32]:
hourly_export = hourly.drop(columns=['timestamp_ny'], errors='ignore')
hourly_path = PROCESSED / 'usdjpy_hourly_modelday_returns.csv'
daily_path = PROCESSED / 'usdjpy_daily_model_panel.csv'
events_path = PROCESSED / 'g10_usd_jpy_events_modelday_mapped.csv'

def write_csv_with_retry(frame, path, attempts=3):
    import time
    for attempt in range(attempts):
        try:
            frame.to_csv(path, index=False)
            return
        except OSError:
            if attempt + 1 == attempts:
                raise
            time.sleep(1)

write_csv_with_retry(hourly_export, hourly_path)
write_csv_with_retry(panel, daily_path)
write_csv_with_retry(events_mapped, events_path)
final_summary = pd.Series({
    'model_sessions': len(panel),
    'rv_eligible_days': int(panel.rv_eligible.sum()),
    'rv_eligible_pct': float(panel.rv_eligible.mean()),
    'iv_available_days': int(panel.iv_available.sum()),
    'current_joint_panel_eligible_days': int(panel.joint_panel_eligible.sum()),
    'forecast_pair_eligible_days': int(panel.forecast_pair_eligible.sum()),
    'current_joint_lost_only_next_rv_unavailable': int((panel.joint_panel_eligible & ~panel.forecast_pair_eligible).sum()),
    'rv_ineligible_days_with_raw_rv': int((~panel.rv_eligible & panel.rv_within_raw.notna()).sum()),
    'eligible_monday_sessions': int(panel.eligible_monday_session.sum()),
    'weekend_reopening_gaps_constructed': int((panel.closure_gap_type == 'weekend').sum()),
    'weekend_reopening_gaps_missing': int((panel.eligible_monday_session & panel.closure_gap_type.ne('weekend')).sum()),
    'rv_only_days': int((panel.rv_eligible & ~panel.iv_available).sum()),
    'iv_only_days': int((~panel.rv_eligible & panel.iv_available).sum()),
    'rv_iv_both_days': int((panel.rv_eligible & panel.iv_available).sum()),
    'rv_iv_neither_days': int((~panel.rv_eligible & ~panel.iv_available).sum()),
    'events_in_rv_session': int(events_mapped.event_in_rv_session.sum()),
    'events_market_closure_within_sample': int(events_mapped.event_in_market_closure.sum()),
    'events_outside_model_sample': int(events_mapped.event_outside_model_sample.sum()),
    'events_other_unresolved': int((events_mapped.event_status == 'other/unresolved').sum()),
    'cut_to_cut_max_abs_difference': float(cut_stats['max_abs_difference']),
    'cut_to_cut_p99_abs_difference': float(cut_stats['p99_abs_difference']),
})
print(final_summary.to_string())
print('Regenerated:', hourly_path.name, daily_path.name, events_path.name)

model_sessions                                 6.044000e+03
rv_eligible_days                               5.764000e+03
rv_eligible_pct                                9.536731e-01
iv_available_days                              5.719000e+03
current_joint_panel_eligible_days              5.445000e+03
forecast_pair_eligible_days                    5.372000e+03
current_joint_lost_only_next_rv_unavailable    7.300000e+01
rv_ineligible_days_with_raw_rv                 7.800000e+01
eligible_monday_sessions                       1.208000e+03
weekend_reopening_gaps_constructed             1.157000e+03
weekend_reopening_gaps_missing                 5.100000e+01
rv_only_days                                   3.190000e+02
iv_only_days                                   2.740000e+02
rv_iv_both_days                                5.445000e+03
rv_iv_neither_days                             6.000000e+00
events_in_rv_session                           1.481300e+04
events_market_closure_within_sample     